# Day 10 · Read the failures, not the averages

**First-principles question:** Why does it fail, and which fix matters most?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import csv
import pandas as pd
import matplotlib.pyplot as plt
from lab.acme import load_cases, load_results
from lab.evals import score

cases = {c["id"]: c for c in load_cases()}
FAILURES = Path("failures.csv")    # in this folder

## 1 · Collect every failure into a spreadsheet

In [ ]:
rows = []
for version in ["A", "B", "C"]:
    for r in load_results(version):
        s = score(cases[r["id"]], r)
        if not s["passed"]:
            failed = [k for k in ("intent_ok", "facts_ok", "policy_ok", "human_ok") if not s[k]]
            rows.append({"key": f"{r['id']}:{version}", "case": r["id"], "version": version,
                         "failed_checks": " ".join(failed), "violations": " ".join(s["violations"]),
                         "note": "", "category": ""})

if FAILURES.exists():   # keep the notes you already wrote
    old = {r["key"]: r for r in csv.DictReader(FAILURES.open(encoding="utf-8"))}
    for r in rows:
        r["note"], r["category"] = old.get(r["key"], {}).get("note", ""), old.get(r["key"], {}).get("category", "")
with FAILURES.open("w", newline="", encoding="utf-8") as fh:
    writer = csv.DictWriter(fh, fieldnames=list(rows[0]))
    writer.writeheader(); writer.writerows(rows)
print(len(rows), "failures written to", FAILURES.resolve())

## 2 · A tiny viewer
Read one failure at a time. Then write a short **note** and a **category** for it in `failures.csv` (VS Code or Numbers). Re-run step 1 any time: your notes are kept.

In [ ]:
def show(key):
    case_id, version = key.split(":")
    case = cases[case_id]
    out = next(r for r in load_results(version) if r["id"] == case_id)
    print(f"── {key} · expected {case['expected_intent']} · must include {case['must_include']} · "
          f"needs_human {case['needs_human']}")
    print("EMAIL:", case["message"])
    print(f"OUTPUT: intent={out.get('intent')} needs_human={out.get('needs_human')}\n{out.get('reply')}")
    print("CHECKS:", score(case, out))

show(rows[0]["key"])

## 3 · Count by category (after you have written your notes)

In [ ]:
notes = pd.read_csv(FAILURES).fillna("")
labelled = notes[(notes.note != "") & (notes.category != "")]
pivot = pd.crosstab(labelled.category, labelled.version, margins=True)
pivot.drop("All").drop(columns="All").plot.barh(stacked=True, figsize=(7, 3), title="Failures by category")
plt.show()
pivot

**Top 3 categories, each with one example case id:**

1. …
2. …
3. …

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert len(labelled) >= 20, f"Label at least 20 failures (note + category): {len(labelled)} so far"
n_categories = labelled.category.nunique()
assert 4 <= n_categories <= 7, f"Use 4 to 7 categories (you have {n_categories})"
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")